# 05 · Selección del ganador, prueba única y Model Registry

Para cada horizonte:
1. Ganador y segundo mejor según la CV del tuning (media ± desviación y diferencia pareada por pliegue).
2. Ambos se evalúan **una sola vez** en el hold-out (últimos 6 meses), nunca visto hasta aquí.
3. Métrica de negocio: días de alta demanda **no anticipados** (FN) frente a la regla del PSet 1,
   con un umbral elegido en la CV (no en prueba): el de mayor recall cuya precisión es al menos la
   de la regla de negocio.
4. Costo (tiempo de entrenamiento/inferencia) y explicabilidad (importancia por permutación).
5. Registro: `ecu911-t{h}-modelo` con alias `champion` (ganador) y `challenger` (segundo).

In [ ]:
import os, sys, json, time, tempfile, pathlib
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1"); os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "true")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
os.environ.setdefault('TQDM_DISABLE', '1')
from mlflow import MlflowClient
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, precision_recall_curve

from src.datos import cargar_obt, construir_tabla_modelado, HORIZONTES, COLUMNAS_X, SEED, RAIZ
from src.validacion import separar_holdout, pliegues_temporales, tabla_p90, etiquetar, p90_objetivo
from src.experimento import (configurar_mlflow, evaluar_regla, score_regla_negocio, umbral_negocio,
                             metricas_negocio, registrar_corrida)
from src.reporte import guardar_fig, guardar_tabla

SINTETICO = os.getenv("ECU911_SINTETICO") == "1"
sns.set_theme(style="whitegrid")
obt, meta = cargar_obt()
configurar_mlflow()
cliente = MlflowClient()
dir_res = RAIZ / ("data/dev/resultados" if SINTETICO else "docs/resultados")
tuning = pd.read_csv(dir_res / "tuning_resumen.csv")
tuning[["h", "modelo", "pr_auc_default", "pr_auc_tuned", "std_tuned", "mejora", "mejora_cuenta"]].round(4)

## 1. Ganador vs. segundo mejor en CV

In [ ]:
def oof_de(run_id):
    d = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="oof.parquet")
    return pd.read_parquet(d)

def pr_auc_por_pliegue(oof):
    return oof.groupby("fold").apply(lambda g: average_precision_score(g.y, g.score), include_groups=False)

seleccion, comparaciones = {}, []
for h in HORIZONTES:
    t = tuning[tuning.h == h].sort_values("pr_auc_tuned", ascending=False).reset_index(drop=True)
    g, s = t.iloc[0], t.iloc[1]
    fg, fs = pr_auc_por_pliegue(oof_de(g.run_id_padre)), pr_auc_por_pliegue(oof_de(s.run_id_padre))
    dif = fg - fs
    seleccion[h] = {"ganador": g, "segundo": s}
    comparaciones.append({"h": h, "ganador": g.modelo, "segundo": s.modelo,
                          "cv_ganador": f"{fg.mean():.4f} ± {fg.std(ddof=0):.4f}",
                          "cv_segundo": f"{fs.mean():.4f} ± {fs.std(ddof=0):.4f}",
                          "diferencia_media": dif.mean(), "std_diferencia_pareada": dif.std(ddof=0),
                          "pliegues_que_gana": int((dif > 0).sum()),
                          "supera_variacion_entre_pliegues": bool(dif.mean() > max(fg.std(ddof=0), fs.std(ddof=0)))})
comp = pd.DataFrame(comparaciones)
guardar_tabla(comp.round(5), "seleccion_cv")
comp.round(4)

### ¿Y TimesFM?

TimesFM (zero-shot) tuvo la mejor media entre los candidatos por defecto, pero no entra al tuning
(no tiene hiperparámetros entrenables) ni al registro como pipeline de scikit-learn. Se compara aquí,
pliegue a pliegue, contra el ganador ajustado, usando sus predicciones fuera de pliegue.

In [ ]:
candidatos = pd.read_csv(dir_res / "candidatos_resultados.csv")
filas_tfm = []
for h in HORIZONTES:
    run_tfm = candidatos[(candidatos.h == h) & (candidatos.familia == "state_of_the_art")].run_id.iloc[0]
    ft = pr_auc_por_pliegue(oof_de(run_tfm))
    fg = pr_auc_por_pliegue(oof_de(seleccion[h]["ganador"].run_id_padre))
    d = fg - ft
    filas_tfm.append({"h": h, "ganador": seleccion[h]["ganador"].modelo,
                      "cv_ganador": f"{fg.mean():.4f} ± {fg.std(ddof=0):.4f}",
                      "cv_timesfm": f"{ft.mean():.4f} ± {ft.std(ddof=0):.4f}",
                      "diferencia_media": d.mean(), "std_diferencia_pareada": d.std(ddof=0),
                      "pliegues_que_gana_el_ganador": int((d > 0).sum())})
tfm = pd.DataFrame(filas_tfm)
guardar_tabla(tfm.round(5), "seleccion_vs_timesfm")
tfm.round(4)

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONTES), figsize=(12, 3.8))
for ax, h in zip(axes, HORIZONTES):
    for rol, m in (("ganador", "o-"), ("segundo", "s--")):
        f = pr_auc_por_pliegue(oof_de(seleccion[h][rol].run_id_padre))
        ax.plot(f.index, f.values, m, label=f"{rol}: {seleccion[h][rol].modelo}")
    ax.set(title=f"PR-AUC por pliegue (h = {h})", xlabel="pliegue", ylabel="PR-AUC"); ax.legend(fontsize=8)
plt.tight_layout(); guardar_fig(fig, "seleccion_pliegues"); plt.show()

## 2. Evaluación única en el hold-out + métrica de negocio

In [ ]:
finales, curvas, estratos_tabla = [], {}, []
from src.validacion import inicio_holdout
_inicio = inicio_holdout(construir_tabla_modelado(obt, 3))
vol = obt[obt.fecha < _inicio].groupby("cod_canton").n_emergencias.mean()   # estratos con datos de train
estrato = pd.qcut(vol.rank(method="first"), 4, labels=["Q1 bajo", "Q2", "Q3", "Q4 alto"])

for h in HORIZONTES:
    tabla = construir_tabla_modelado(obt, h)
    train, test = separar_holdout(tabla, h)
    pl = pliegues_temporales(train, h)
    p90 = tabla_p90(train)                       # P90 con TODO el entrenamiento
    y_test = etiquetar(test, p90)

    # Regla de negocio: precisión en CV (fija el piso de precisión) y resultado en prueba
    r_neg = evaluar_regla(score_regla_negocio, train, h, pl)
    neg_cv = metricas_negocio(r_neg.oof.y, r_neg.oof.score, 1e-12)        # alerta si score > 0
    s_neg_test = score_regla_negocio(test.prom_4_mismo_dia, p90_objetivo(test, p90))
    neg_test = metricas_negocio(y_test, s_neg_test, 1e-12)
    curvas[(h, "Regla de negocio")] = (y_test, s_neg_test)

    for rol in ("ganador", "segundo"):
        fila = seleccion[h][rol]
        pipe = mlflow.sklearn.load_model(f"runs:/{fila.run_id_padre}/modelo")   # entrenado con todo train
        oof = oof_de(fila.run_id_padre)
        umbral = umbral_negocio(oof.y.to_numpy(), oof.score.to_numpy(), precision_min=neg_cv["precision"])
        t0 = time.perf_counter()
        s_test = pipe.predict_proba(test[COLUMNAS_X])[:, 1]
        seg_inf = time.perf_counter() - t0
        pr_test = average_precision_score(y_test, s_test)
        m = metricas_negocio(y_test, s_test, umbral)
        m70 = metricas_negocio(y_test, s_test, 0.70)
        reduccion_fn = (neg_test["fn"] - m["fn"]) / neg_test["fn"]
        curvas[(h, fila.modelo)] = (y_test, s_test)
        for e in estrato.cat.categories:
            idx = test.cod_canton.map(estrato).eq(e).to_numpy()
            estratos_tabla.append({"h": h, "modelo": fila.modelo, "estrato": e, "prevalencia": y_test[idx].mean(),
                                   "pr_auc_test": average_precision_score(y_test[idx], s_test[idx])})
        run_final = registrar_corrida(
            f"final_{rol}_{fila.modelo}_t{h}", "final", fila.familia, h, json.loads(fila.mejores_params), None, meta,
            pipeline=pipe, X_ejemplo=train[COLUMNAS_X].sample(200, random_state=SEED),
            metricas_extra={"pr_auc_test": pr_test, "pr_auc_cv_mean": fila.pr_auc_tuned, "pr_auc_cv_std": fila.std_tuned,
                            "umbral_negocio": umbral, "precision_min_negocio": neg_cv["precision"],
                            "test_tp": m["tp"], "test_fp": m["fp"], "test_fn": m["fn"], "test_precision": m["precision"],
                            "test_recall": m["recall"], "test_alertas": m["alertas"],
                            "baseline_negocio_test_fn": neg_test["fn"], "baseline_negocio_test_precision": neg_test["precision"],
                            "reduccion_fn_vs_negocio": reduccion_fn, "test_recall_umbral_070": m70["recall"],
                            "test_precision_umbral_070": m70["precision"],
                            "segundos_inferencia_test": seg_inf, "segundos_entrenamiento": fila.segundos_entrenamiento / 5},
            tags_extra={"rol": "champion" if rol == "ganador" else "challenger", "run_tuning": fila.run_id_padre})
        finales.append({"h": h, "rol": rol, "modelo": fila.modelo, "run_id_final": run_final,
                        "pr_auc_cv": fila.pr_auc_tuned, "pr_auc_test": pr_test, "umbral": umbral,
                        "precision_test": m["precision"], "recall_test": m["recall"], "fn_test": m["fn"],
                        "alertas_test": m["alertas"], "fn_regla_negocio": neg_test["fn"],
                        "precision_regla_negocio": neg_test["precision"], "recall_regla_negocio": neg_test["recall"],
                        "reduccion_fn_vs_negocio": reduccion_fn, "recall_umbral_070": m70["recall"],
                        "precision_umbral_070": m70["precision"],
                        "seg_entrenamiento_por_ajuste": fila.segundos_entrenamiento / 5,
                        "ms_inferencia_por_1000_filas": seg_inf / len(test) * 1e6})
    print(f"h={h}: positivos en prueba {y_test.sum():,} de {len(y_test):,} ({y_test.mean():.2%}); "
          f"regla de negocio en prueba: precisión {neg_test['precision']:.3f}, recall {neg_test['recall']:.3f}, FN {neg_test['fn']:,}")

finales = pd.DataFrame(finales)
guardar_tabla(finales.round(5), "seleccion_final")
finales.drop(columns="run_id_final").round(4)

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONTES), figsize=(13, 4.5))
for ax, h in zip(axes, HORIZONTES):
    for (hh, nombre), (y, s) in curvas.items():
        if hh != h: continue
        p, r, _ = precision_recall_curve(y, s)
        ax.plot(r, p, label=f"{nombre} (PR-AUC {average_precision_score(y, s):.3f})")
    ax.axhline(y.mean(), ls=":", c="gray", label="azar")
    for f in finales[finales.h == h].itertuples():
        ax.plot(f.recall_test, f.precision_test, "k*", ms=12)
    ax.set(title=f"Curva precisión-recall en el hold-out (h = {h})", xlabel="recall", ylabel="precisión")
    ax.legend(fontsize=8)
plt.tight_layout(); guardar_fig(fig, "seleccion_curvas_pr"); plt.show()

### Comparación con el mismo número de alertas que la regla de negocio

El umbral elegido en CV no garantiza la misma precisión en prueba (la demanda cambia con el tiempo).
Para comparar sin ese efecto, cada modelo emite exactamente tantas alertas como la regla de negocio
(las de mayor score) y se cuentan los días de alta demanda que anticipa. No elige nada con la prueba:
el número de alertas lo fija la regla.

In [ ]:
presupuesto = []
for h in HORIZONTES:
    y, s_reg = curvas[(h, "Regla de negocio")]
    k = int((s_reg > 0).sum())
    for (hh, nombre), (yy, s) in curvas.items():
        if hh != h: continue
        top = np.argsort(-np.asarray(s), kind="stable")[:k]
        tp = int(yy[top].sum())
        presupuesto.append({"h": h, "modelo": nombre, "alertas": k, "anticipados": tp,
                            "no_anticipados": int(yy.sum() - tp), "precision": tp / k, "recall": tp / yy.sum()})
presupuesto = pd.DataFrame(presupuesto)
base = presupuesto[presupuesto.modelo == "Regla de negocio"].set_index("h").no_anticipados
presupuesto["reduccion_fn_vs_negocio"] = 1 - presupuesto.no_anticipados / presupuesto.h.map(base)
guardar_tabla(presupuesto.round(4), "seleccion_mismo_presupuesto")
presupuesto.round(4)

In [ ]:
est = pd.DataFrame(estratos_tabla)
guardar_tabla(est.round(4), "seleccion_estratos")
est.pivot_table(index=["h", "modelo"], columns="estrato", values="pr_auc_test", observed=True).round(3)

## 3. Explicabilidad: importancia por permutación (ganador, hold-out)

In [ ]:
imp_tabla = []
for h in HORIZONTES:
    tabla = construir_tabla_modelado(obt, h)
    train, test = separar_holdout(tabla, h)
    y_test = etiquetar(test, tabla_p90(train))
    fila = seleccion[h]["ganador"]
    pipe = mlflow.sklearn.load_model(f"runs:/{fila.run_id_padre}/modelo")
    muestra = test.sample(min(20_000, len(test)), random_state=SEED)
    ym = etiquetar(muestra, tabla_p90(train))
    pi = permutation_importance(pipe, muestra[COLUMNAS_X], ym, scoring="average_precision",
                                n_repeats=3, random_state=SEED, n_jobs=1)
    imp_tabla.append(pd.DataFrame({"h": h, "modelo": fila.modelo, "columna": COLUMNAS_X,
                                   "importancia": pi.importances_mean, "std": pi.importances_std}))
imp = pd.concat(imp_tabla, ignore_index=True).sort_values(["h", "importancia"], ascending=[True, False]).reset_index(drop=True)
guardar_tabla(imp.round(5), "seleccion_importancia")
g = sns.catplot(data=imp.groupby("h").head(12).reset_index(drop=True), x="importancia", y="columna", col="h", kind="bar",
                sharey=False, height=4, aspect=1.2)
g.set_axis_labels("caída de PR-AUC al permutar", "")
guardar_fig(g.figure, "seleccion_importancia"); plt.show()

## 4. Registro en el Model Registry

In [ ]:
registro = []
for h in HORIZONTES:
    nombre = f"{'dev-' if SINTETICO else ''}ecu911-t{h}-modelo"
    for rol, alias in (("ganador", "champion"), ("segundo", "challenger")):
        run_id = finales[(finales.h == h) & (finales.rol == rol)].run_id_final.iloc[0]
        v = mlflow.register_model(f"runs:/{run_id}/modelo", nombre)
        cliente.set_registered_model_alias(nombre, alias, v.version)
        cliente.set_model_version_tag(nombre, v.version, "horizonte", f"t{h}")
        registro.append({"modelo_registrado": nombre, "alias": alias, "version": v.version,
                         "run_id": run_id, "modelo": seleccion[h][rol].modelo})
registro = pd.DataFrame(registro)
guardar_tabla(registro, "registro")
registro

## 5. Decisión

| | t+3 | t+7 |
|---|---|---|
| **Champion** | HistGradientBoosting · `ecu911-t3-modelo` v1 · run `b971b96e…` | HistGradientBoosting · `ecu911-t7-modelo` v1 · run `13965788…` |
| Challenger | MLP · v2 · run `05b8d7e8…` | MLP · v2 · run `283bf470…` |
| PR-AUC CV (champion) | 0.185 ± 0.054 | 0.185 ± 0.056 |
| **PR-AUC prueba** (champion / challenger / regla) | **0.258** / 0.251 / — | **0.257** / 0.247 / — |
| FN en prueba: regla → champion (umbral de CV) | 2,617 → 2,318 (**−11.4 %**) | 2,570 → 2,360 (−8.2 %) |
| Precisión en prueba: regla / champion | 0.382 / 0.318 | 0.386 / 0.357 |
| Con el **mismo número de alertas** que la regla | −1.2 % FN (619 vs 588 anticipados) | −1.3 % FN (615 vs 583) |

**Por qué gana HistGradientBoosting:**
1. **CV:** mejor media en ambos horizontes y gana a la MLP en 4 de 5 pliegues, pero la diferencia (+0.009 / +0.008) **no supera la variación entre pliegues**: en CV es un empate técnico. Lo mismo frente a TimesFM (+0.001 / +0.005, gana 3 de 5 pliegues).
2. **Prueba (una sola vez):** también es el mejor (0.258 vs 0.251 y 0.257 vs 0.247), y mejora más en los cantones de bajo volumen (estrato Q1: 0.196 vs 0.173).
3. **Costo y explicabilidad:** entrena en ~3 s y puntúa 1,000 filas en ~4 ms; la importancia por permutación es estable y legible (nivel reciente relativo al P90 y feriado de t+h). TimesFM, con una CV equivalente, necesita 200 M de parámetros, 128 días de historia cruda por cantón y ~4 ms **por serie**, y no se puede explicar ni reentrenar: ante un empate se elige el modelo más barato y explicable.

**Métrica de negocio, con honestidad:** con el umbral elegido en CV, el champion cumple el objetivo del PSet 1 (≥ 10 % menos días no anticipados) **en t+3, no en t+7**. Pero lo logra emitiendo ~80 % más alertas que la regla, porque la precisión que se fijó en CV no se sostuvo en prueba (deriva temporal). **A igual número de alertas, la mejora sobre la regla de negocio es pequeña (≈ 1.2 %)**. El valor operativo del modelo está en poder **elegir el punto de la curva** (más recall a cambio de más alertas), algo que la regla fija no permite. El umbral 0.70 del PSet 1 no sirve: ninguna probabilidad pasa de ≈ 0.6, el recall es 0.